In [1]:
%pip install torch torchvision

Note: you may need to restart the kernel to use updated packages.


In [3]:
import ssl
import certifi
ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from pathlib import Path

# -----------------------
# 1. Paths (IMPORTANT)
# -----------------------
BASE_DIR = Path("../../data/dataset/archive/original")  
# from src/garbage-classifier/train.ipynb → go up 2 levels

print("Dataset path:", BASE_DIR.resolve())

# -----------------------
# 2. Device (M3 GPU = MPS)
# -----------------------
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

# -----------------------
# 3. Transforms
# -----------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# -----------------------
# 4. Dataset
# -----------------------
dataset = datasets.ImageFolder(root=str(BASE_DIR), transform=transform)

class_names = dataset.classes
num_classes = len(class_names)

print("Classes:", class_names)
print("Num classes:", num_classes)

loader = DataLoader(dataset, batch_size=32, shuffle=True)

# -----------------------
# 5. Model (MobileNetV2)
# -----------------------
model = models.mobilenet_v2(weights="IMAGENET1K_V1")

# freeze feature extractor
for param in model.features.parameters():
    param.requires_grad = False

# replace classifier head
model.classifier = nn.Sequential(
    nn.Linear(model.last_channel, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, num_classes)
)

model = model.to(device)

# -----------------------
# 6. Loss + Optimizer
# -----------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

# -----------------------
# 7. Training loop
# -----------------------
epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss:.4f}")

# -----------------------
# 8. Save model
# -----------------------
torch.save({
    "model_state": model.state_dict(),
    "class_names": class_names
}, "garbage_model.pth")

print("Model saved as garbage_model.pth")

Dataset path: /Users/mihai/Developer/garbage-optimizer/data/dataset/archive/original
Using device: mps
Classes: ['battery', 'biological', 'cardboard', 'clothes', 'glass', 'metal', 'paper', 'plastic', 'shoes', 'trash']
Num classes: 10
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /Users/mihai/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100.0%


Epoch 1/5 | Loss: 227.0674
Epoch 2/5 | Loss: 146.3936
Epoch 3/5 | Loss: 127.9821
Epoch 4/5 | Loss: 118.1506
Epoch 5/5 | Loss: 110.0609
Model saved as garbage_model.pth
